# Pandas for AI & Machine Learning — Beginner-to-Practical Notebook

**Learn by running each cell from top to bottom.** This notebook teaches the Pandas operations most useful for preparing, exploring, cleaning, and transforming tabular data for machine learning. It intentionally excludes rarely needed Pandas APIs.

**You will learn:** DataFrames and Series → importing data → inspecting → selecting → filtering → cleaning → grouping → combining → time features → ML-ready preprocessing.

**Prerequisites:** Basic Python. **Setup:** `pip install pandas numpy` (run in a terminal if needed). No internet or external dataset required.

> **Important ML habit:** Split data into training and test sets before learning imputation statistics, category vocabularies, or scaling parameters. This notebook demonstrates Pandas preprocessing; for real projects use scikit-learn pipelines to prevent leakage.

## 1. Import Pandas and create a reusable dataset

`pd` is the conventional alias. We will reuse this small synthetic student dataset throughout the notebook.

In [ ]:
import pandas as pd
import numpy as np

raw = {
    "student_id": [101, 102, 103, 104, 105, 106, 107, 108],
    "age": [20, 21, 22, np.nan, 20, 23, 21, 22],
    "study_hours": [2.5, 4.0, 3.0, 5.0, np.nan, 1.5, 4.5, 3.5],
    "score": [65, 82, 74, 91, 68, 55, 88, 78],
    "city": ["Pune", "Mumbai", "Pune", "Delhi", "Mumbai", "Pune", None, "Delhi"],
    "passed": [1, 1, 1, 1, 1, 0, 1, 1],
    "exam_date": ["2026-01-05", "2026-01-06", "2026-01-07", "2026-01-08",
                  "2026-01-09", "2026-01-10", "2026-01-11", "2026-01-12"]
}
df = pd.DataFrame(raw)
df

## 2. Series, DataFrame, and basic creation

A **Series** is a one-dimensional labeled array; a **DataFrame** is a two-dimensional labeled table. Use `pd.Series()`, `pd.DataFrame()`, and `pd.Index()` when building or examining data.

In [ ]:
s = pd.Series([10, 20, 30], name="values")
small = pd.DataFrame({"feature": [1, 2], "target": [0, 1]})
print(s)
print(small)
print(df.index, df.columns)

## 3. Read and write datasets

Common inputs: `pd.read_csv()`, `pd.read_excel()`, `pd.read_json()`, `pd.read_parquet()`, `pd.read_sql()`; outputs: `to_csv()`, `to_excel()`, `to_json()`, `to_parquet()`. Excel/Parquet/SQL may require optional drivers. Below, CSV works without extra files or services.

In [1]:
from io import StringIO
csv_text = "feature,target\n1.2,0\n2.4,1\n3.1,0"
csv_df = pd.read_csv(StringIO(csv_text))
print(csv_df)
print(csv_df.to_csv(index=False))
# Real files: pd.read_csv("train.csv"); df.to_csv("clean.csv", index=False)
# Excel: pd.read_excel("train.xlsx"); df.to_excel("clean.xlsx", index=False)
# JSON: pd.read_json("data.json"); df.to_json("data.json", orient="records")
# Parquet: pd.read_parquet("data.parquet"); df.to_parquet("data.parquet")
# SQL: pd.read_sql("SELECT * FROM students", connection)

NameError: name 'pd' is not defined

## 4. First look: shape, head, tail, info

Start every new dataset with `.head()`, `.tail()`, `.sample()`, `.shape`, `.columns`, `.dtypes`, `.info()`, and `.describe()`. Properties such as `.shape` have **no parentheses**.

In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Types:\n", df.dtypes)
print("First rows:\n", df.head(3))
print("Last rows:\n", df.tail(2))
print("Random rows:\n", df.sample(3, random_state=42))
df.info()
print(df.describe(include="all"))

## 5. Select columns and rows

Use `df["column"]` for a Series, `df[["a", "b"]]` for a DataFrame, `.loc[]` for **labels**, `.iloc[]` for **integer positions**. `.at[]` and `.iat[]` retrieve individual values.

In [ ]:
print(df["score"].head())
print(df[["age", "score"]].head())
print(df.loc[0:2, ["city", "score"]])  # loc label slices include endpoint
print(df.iloc[0:3, 1:4])               # iloc endpoint excluded
print(df.at[0, "score"], df.iat[0, 3])

## 6. Filter rows with conditions

Boolean masks are central to data cleaning. Combine conditions with `&`, `|`, `~` and wrap each comparison in parentheses. `.isin()`, `.between()`, `.query()` and `.notna()` simplify filtering.

In [ ]:
print(df[df["score"] >= 75])
print(df[(df["score"] >= 75) & (df["city"] == "Pune")])
print(df[df["city"].isin(["Pune", "Mumbai"])])
print(df[df["score"].between(60, 80)])
print(df.query("score >= 80"))
print(df[df["city"].notna()])

## 7. Sort, count and unique values

`.sort_values()` sorts by features; `.sort_index()` sorts index labels. `.value_counts()` counts categories; `.unique()` lists distinct values; `.nunique()` counts distinct values.

In [ ]:
print(df.sort_values("score", ascending=False).head())
print(df.sort_index(ascending=False).head())
print(df["city"].value_counts(dropna=False))
print(df["city"].unique())
print(df["city"].nunique(dropna=True))

## 8. Numerical summaries and relationships

Useful aggregations: `.count()`, `.sum()`, `.mean()`, `.median()`, `.min()`, `.max()`, `.std()`, `.var()`, `.quantile()`, `.agg()`, and `.corr()`. `corr()` helps explore linear associations but does not establish causation.

In [ ]:
numeric = df.select_dtypes(include="number")
print(numeric.mean())
print(numeric.median())
print(numeric.min(), numeric.max())
print(numeric.std(), numeric.var())
print(numeric.quantile([0.25, 0.5, 0.75]))
print(numeric.agg(["count", "sum", "mean"]))
print(numeric.corr())

## 9. Missing data: detect and quantify

Missing values can appear as `NaN`, `None`, or `NaT`. Use `.isna()` / `.isnull()` and `.notna()` / `.notnull()`; aliases are equivalent. Count missing values per column before deciding how to handle them.

In [ ]:
print(df.isna().sum())
print((df.isna().mean() * 100).round(1))
print(df.notna().sum())

## 10. Missing data: fill and drop

`.fillna()` imputes values; `.dropna()` removes incomplete rows or columns. For ML, compute fill statistics **on training data only** and reuse them for validation/test data. `ffill()` / `bfill()` are mostly useful for ordered/time-series data.

In [ ]:
clean = df.copy()
clean["age"] = clean["age"].fillna(clean["age"].median())
clean["study_hours"] = clean["study_hours"].fillna(clean["study_hours"].median())
clean["city"] = clean["city"].fillna("Unknown")
print(clean.isna().sum())
print(df.dropna().shape)
print(df.dropna(subset=["score", "city"]).shape)
print(pd.Series([1, np.nan, 3]).ffill())
print(pd.Series([1, np.nan, 3]).bfill())

## 11. Duplicates and data consistency

`.duplicated()` marks duplicates; `.drop_duplicates()` removes them. `.replace()` standardizes inconsistent labels. `.rename()` improves column names. `.astype()` changes types; `.convert_dtypes()` infers nullable Pandas types.

In [ ]:
dupes = pd.concat([df, df.iloc[[0]]], ignore_index=True)
print(dupes.duplicated().sum())
print(dupes.drop_duplicates().shape)
fixed = clean.replace({"city": {"Pune": "PUNE"}}).rename(columns={"score": "exam_score"})
fixed["passed"] = fixed["passed"].astype("int64")
print(fixed.convert_dtypes().dtypes)

## 12. Create and transform features

Use column arithmetic, `.assign()`, `.map()`, `.apply()`, `.clip()`, `.round()`, and `.where()` for feature engineering. Prefer **vectorized** operations over `.apply()` for speed when possible.

In [ ]:
features = clean.assign(
    hours_per_age=clean["study_hours"] / clean["age"],
    score_band=np.where(clean["score"] >= 80, "high", "other")
)
features["city_code"] = features["city"].map({"Pune": 1, "Mumbai": 2, "Delhi": 3, "Unknown": 0})
features["score_clipped"] = features["score"].clip(lower=60, upper=90)
features["hours_rounded"] = features["study_hours"].round(0)
features["bonus"] = features["score"].apply(lambda x: 5 if x >= 85 else 0)
features["adjusted"] = features["score"].where(features["score"] >= 60, 60)
print(features.head())

## 13. String cleaning

`.str` methods operate on text columns. Common methods include `.str.strip()`, `.str.lower()`, `.str.upper()`, `.str.contains()`, `.str.replace()`, `.str.split()`, and `.str.len()`. Fill missing strings first when necessary.

In [ ]:
cities = pd.Series([" Pune ", "MUMBAI", "New-Delhi", None], dtype="string")
normalized = cities.str.strip().str.lower().str.replace("-", " ", regex=False)
print(normalized)
print(normalized.str.contains("delhi", na=False))
print(normalized.str.split(" "))
print(normalized.str.len())

## 14. Dates and time features

Use `pd.to_datetime()` to parse dates, then `.dt.year`, `.dt.month`, `.dt.day`, `.dt.dayofweek`, `.dt.hour` as appropriate. Invalid dates can be converted to `NaT` using `errors="coerce"`.

In [ ]:
dated = clean.copy()
dated["exam_date"] = pd.to_datetime(dated["exam_date"], errors="coerce")
dated["exam_month"] = dated["exam_date"].dt.month
dated["exam_weekday"] = dated["exam_date"].dt.dayofweek
print(dated[["exam_date", "exam_month", "exam_weekday"]].head())

## 15. Grouping and aggregation

`.groupby()` computes per-group summaries; `.agg()` applies multiple aggregations; `.transform()` broadcasts group statistics back to each row. `.pivot_table()` builds aggregated cross-tabulations; `pd.crosstab()` counts combinations.

In [ ]:
print(clean.groupby("city")["score"].mean())
print(clean.groupby("city").agg(avg_score=("score", "mean"), n=("score", "size")))
clean["city_mean_score"] = clean.groupby("city")["score"].transform("mean")
print(clean[["city", "score", "city_mean_score"]])
print(pd.pivot_table(clean, values="score", index="city", columns="passed", aggfunc="mean"))
print(pd.crosstab(clean["city"], clean["passed"]))

## 16. Combine datasets

`pd.concat()` stacks rows or columns; `pd.merge()` joins tables by keys; `.join()` joins primarily by index. Check join cardinality with `validate=` when appropriate to catch unexpected row multiplication.

In [ ]:
students = clean[["student_id", "score"]]
clubs = pd.DataFrame({"student_id": [101, 102, 103], "club": ["AI", "Robotics", "AI"]})
print(pd.merge(students, clubs, on="student_id", how="left", validate="one_to_one"))
print(pd.concat([students.iloc[:3], students.iloc[3:]], ignore_index=True).shape)
print(students.set_index("student_id").join(clubs.set_index("student_id"), how="left").head())

## 17. Index and reshape

`.set_index()`, `.reset_index()`, `.melt()`, and `.pivot()` are useful for changing how tables are organized. `pivot()` requires unique index/column pairs; use `pivot_table()` when aggregation is needed.

In [ ]:
indexed = clean.set_index("student_id")
print(indexed.head(2))
print(indexed.reset_index().head(2))
wide = pd.DataFrame({"id": [1, 2], "math": [70, 80], "science": [75, 85]})
long = wide.melt(id_vars="id", var_name="subject", value_name="mark")
print(long)
print(long.pivot(index="id", columns="subject", values="mark"))

## 18. Encode categorical variables

`pd.get_dummies()` creates one-hot indicator columns. `pd.Categorical()` can represent category data; `.cat.codes` produces integer codes but **those codes are not meaningful numerical distances**. Fit a stable encoder on training data for production.

In [ ]:
encoded = pd.get_dummies(clean[["city", "study_hours"]], columns=["city"], dtype=int)
print(encoded.head())
category = pd.Categorical(clean["city"])
print(category.categories)
print(category.codes)

## 19. Outliers and basic feature scaling

`.quantile()` supports IQR outlier detection. Pandas arithmetic can implement min-max scaling or z-scores for learning. **Do not fit scaling statistics on the entire dataset before splitting.**

In [ ]:
q1, q3 = clean["score"].quantile([0.25, 0.75])
iqr = q3 - q1
outliers = clean[(clean["score"] < q1 - 1.5 * iqr) | (clean["score"] > q3 + 1.5 * iqr)]
print("Outliers:
", outliers)
score = clean["score"]
minmax = (score - score.min()) / (score.max() - score.min())
zscore = (score - score.mean()) / score.std(ddof=0)
print(pd.DataFrame({"minmax": minmax, "zscore": zscore}).head())

## 20. Sampling and quick plotting

`.sample()` selects random rows. `.plot()` and `.hist()` give quick exploratory visualizations (Matplotlib required for plots). Use `random_state` for reproducibility.

In [ ]:
print(clean.sample(frac=0.5, random_state=42)[["student_id", "score"]])
# Uncomment if matplotlib is installed and plots are desired:
# import matplotlib.pyplot as plt
# clean["score"].hist(bins=5)
# plt.show()
# clean.plot.scatter(x="study_hours", y="score")
# plt.show()

## 21. A leakage-safe mini ML preprocessing example

Split first, learn the median and one-hot column layout from training data, then apply the same choices to the test set. This is a teaching example; for real ML workflows, prefer `train_test_split`, `SimpleImputer`, `OneHotEncoder`, and `ColumnTransformer` from scikit-learn.

In [ ]:
dataset = df[["age", "study_hours", "city", "passed"]].copy()
train = dataset.iloc[:6].copy()
test = dataset.iloc[6:].copy()
X_train, y_train = train.drop(columns="passed"), train["passed"]
X_test, y_test = test.drop(columns="passed"), test["passed"]

numeric_cols = ["age", "study_hours"]
train_medians = X_train[numeric_cols].median()
X_train[numeric_cols] = X_train[numeric_cols].fillna(train_medians)
X_test[numeric_cols] = X_test[numeric_cols].fillna(train_medians)
X_train["city"] = X_train["city"].fillna("Unknown")
X_test["city"] = X_test["city"].fillna("Unknown")
X_train = pd.get_dummies(X_train, columns=["city"], dtype=int)
X_test = pd.get_dummies(X_test, columns=["city"], dtype=int)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
print("Train:", X_train.shape, "Test:", X_test.shape)
print(X_train.head())
print(X_test.head())

## 22. Essential Pandas cheat sheet

| Goal | API | Quick syntax |
|---|---|---|
| Create tables | `DataFrame`, `Series` | `pd.DataFrame({...})` |
| Import/export | `read_csv`, `to_csv` | `pd.read_csv(path)` |
| Other formats | `read_excel`, `read_json`, `read_parquet`, `read_sql` | `pd.read_sql(query, conn)` |
| Inspect | `head`, `tail`, `sample`, `info`, `describe` | `df.describe()` |
| Structure | `shape`, `columns`, `dtypes`, `index` | `df.shape` |
| Select | `loc`, `iloc`, `at`, `iat` | `df.loc[mask, cols]` |
| Filter | `isin`, `between`, `query` | `df.query("score > 70")` |
| Sort | `sort_values`, `sort_index` | `df.sort_values("score")` |
| Categories | `unique`, `nunique`, `value_counts` | `df["city"].value_counts()` |
| Statistics | `mean`, `median`, `sum`, `count`, `min`, `max`, `std`, `var`, `quantile`, `corr`, `agg` | `df.select_dtypes("number").corr()` |
| Missing data | `isna`, `notna`, `fillna`, `dropna`, `ffill`, `bfill` | `df.isna().sum()` |
| Clean | `duplicated`, `drop_duplicates`, `replace`, `rename`, `astype`, `convert_dtypes` | `df.drop_duplicates()` |
| Features | `assign`, `map`, `apply`, `clip`, `round`, `where` | `df["x"].clip(0, 10)` |
| Text | `str.strip`, `str.lower`, `str.contains`, `str.replace` | `s.str.lower()` |
| Dates | `to_datetime`, `dt.year`, `dt.month`, `dt.dayofweek` | `pd.to_datetime(s)` |
| Aggregate | `groupby`, `agg`, `transform`, `pivot_table`, `crosstab` | `df.groupby("city").mean(numeric_only=True)` |
| Combine | `concat`, `merge`, `join` | `pd.merge(a, b, on="id")` |
| Reshape | `set_index`, `reset_index`, `melt`, `pivot` | `df.melt(id_vars="id")` |
| Encode | `get_dummies`, `Categorical` | `pd.get_dummies(df)` |
| Quick visuals | `plot`, `hist` | `df["x"].hist()` |

**Note:** This is a curated practical set, not every function in Pandas. It covers the core Pandas skills commonly used in AI/ML data preparation.

## 23. Practice exercises — try before viewing a solution

1. Display only `city`, `study_hours`, and `score` for students scoring at least 75.
2. Calculate the percentage of missing values in every column.
3. Fill missing `study_hours` with its median, without modifying the original DataFrame.
4. Find the average score by city, sorted from highest to lowest.
5. Add a new Boolean feature `high_performer` for scores of 80 or above.
6. Convert the exam date into a weekday number and one-hot encode city.
7. Create a train/test split, compute a training-only median, and apply it to both subsets.

### Suggested solutions

In [ ]:
print(df.loc[df["score"] >= 75, ["city", "study_hours", "score"]])
print(df.isna().mean().mul(100).round(1))
filled_hours = df["study_hours"].fillna(df["study_hours"].median())
print(filled_hours)
print(df.groupby("city")["score"].mean().sort_values(ascending=False))
print(df.assign(high_performer=df["score"] >= 80)[["score", "high_performer"]])
practice = df.copy()
practice["weekday"] = pd.to_datetime(practice["exam_date"]).dt.dayofweek
print(pd.get_dummies(practice, columns=["city"], dtype=int).head())
train_part, test_part = df.iloc[:6].copy(), df.iloc[6:].copy()
median_train = train_part["study_hours"].median()
train_part["study_hours"] = train_part["study_hours"].fillna(median_train)
test_part["study_hours"] = test_part["study_hours"].fillna(median_train)
print(train_part["study_hours"], test_part["study_hours"])

## 24. Next steps for your AI/ML journey

After you can comfortably read, clean, join, group, and encode data with Pandas, move on to **Matplotlib/Seaborn for EDA** and **scikit-learn for preprocessing, train/test splitting, model training, and evaluation**. Practice on one real CSV dataset before trying to memorize more functions.

**Official reference:** https://pandas.pydata.org/docs/user_guide/index.html